In [78]:
import json
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sentence_transformers import SentenceTransformer


# 1. Load your JSON
with open('data/course-api-data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Build a DataFrame of code, title, description
rows = []
for code, info in data.items():
    api = info.get('api_data', {})
    rows.append({
        'courseCode': code,
        'title':       api.get('title',    info.get('title', '')),
        'description': api.get('description', '')
    })
df = pd.DataFrame(rows)

# 3. TF‑IDF → SVD pipeline
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf.fit_transform(df['description'])

svd = TruncatedSVD(n_components=100, random_state=42)

embeddings = svd.fit_transform(X_tfidf)

# 4. Attach & save
df['embedding'] = embeddings.tolist()
df.to_pickle('data/course_embeddings.pkl')
np.save('data/course_embeddings.npy', embeddings)

# 5. (Optional) Quick peek at the table
df[['courseCode','title','description']].head()


# 6. BERT embeddings
embedding_path = 'data/course_bert_embeddings.npy'

if not os.path.exists(embedding_path):
    # Load course descriptions
    df = pd.read_pickle('data/course_embeddings.pkl')  # same df as before

    # Load a BERT-based model (MiniLM is fast + accurate)
    model = SentenceTransformer('all-MiniLM-L6-v2')

    # Generate dense BERT embeddings (384-dim)
    descriptions = df['description'].fillna('').tolist()
    bert_embeddings = model.encode(descriptions, show_progress_bar=True)

    # Save embeddings
    np.save(embedding_path, bert_embeddings)
else:
    print(f"Embeddings file '{embedding_path}' already exists. Skipping embedding generation.")



Embeddings file 'data/course_bert_embeddings.npy' already exists. Skipping embedding generation.


In [79]:
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pandas as pd

# Reload your course DataFrame
df = pd.read_pickle('data/course_embeddings.pkl')  # assumes you re-run the vectorization cell

# 1. Fit TF‑IDF & SVD on your course descriptions
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_tfidf = tfidf.fit_transform(df['description'])

svd = TruncatedSVD(n_components=100, random_state=42)
embeddings = svd.fit_transform(X_tfidf)

# 2. Overwrite embeddings & save models
import numpy as np
df['embedding'] = embeddings.tolist()
df.to_pickle('data/course_embeddings.pkl')
np.save('data/course_embeddings.npy', embeddings)

with open('data/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
with open('data/svd_model.pkl', 'wb') as f:
    pickle.dump(svd, f)

print("Models and embeddings saved.")


Models and embeddings saved.


## Method 1: Direct Cosine‑Similarity Ranking

In [80]:

import pickle, numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

# Load artifacts
df     = pd.read_pickle('data/course_embeddings.pkl')
emb    = np.load('data/course_embeddings.npy')
with open('data/tfidf_vectorizer.pkl','rb') as f: tfidf = pickle.load(f)
with open('data/svd_model.pkl','rb')         as f: svd   = pickle.load(f)

def recommend_cosine(query, top_k=10):
    q_vec = svd.transform(tfidf.transform([query]))               # (1,100)
    sims  = cosine_similarity(emb, q_vec).flatten()              # (N,)
    idxs  = sims.argsort()[::-1][:top_k]
    return df.iloc[idxs][['courseCode','title','description']].assign(similarity=sims[idxs])




## Method 2: Approximate Nearest‑Neighbor Search with FAISS

In [81]:

import pickle, faiss, numpy as np, pandas as pd


# Load & normalize
df  = pd.read_pickle('data/course_embeddings.pkl')
emb = np.load('data/course_embeddings.npy').astype('float32')
faiss.normalize_L2(emb)

# Build or load index
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

with open('data/tfidf_vectorizer.pkl','rb') as f: tfidf = pickle.load(f)
with open('data/svd_model.pkl','rb')         as f: svd   = pickle.load(f)

def recommend_faiss(query, top_k=10):
    q = svd.transform(tfidf.transform([query])).astype('float32')
    faiss.normalize_L2(q)
    D, I = index.search(q, top_k)   # D=sim scores, I=indices
    return df.iloc[I[0]][['courseCode','title','description']].assign(similarity=D[0])




## Method 3: Diversity‑Aware Re‑Ranking (MMR)

In [82]:

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# (Assumes tfidf, svd, emb, df already in scope)

def recommend_mmr(query, top_k=10, λ=0.7):
    q_vec      = svd.transform(tfidf.transform([query])).flatten()
    sims       = cosine_similarity(emb, q_vec.reshape(1,-1)).flatten()
    candidates = list(range(len(sims)))
    selected   = []

    for _ in range(top_k):
        if not selected:
            idx = int(np.argmax(sims))
        else:
            mmr_scores = []
            for i in candidates:
                rel = sims[i]
                red = max(cosine_similarity(emb[selected], emb[i].reshape(1,-1)).flatten())
                mmr_scores.append((i, λ*rel - (1-λ)*red))
            idx = max(mmr_scores, key=lambda x: x[1])[0]
        selected.append(idx)
        candidates.remove(idx)

    return df.iloc[selected][['courseCode','title','description']].assign(similarity=sims[selected])



## Method 4: Graph‑Based Diffusion (Personalized PageRank)

In [83]:

import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# (Assumes tfidf, svd, emb, df already in scope)

def recommend_graph(query, top_k=10):
    q_vec = svd.transform(tfidf.transform([query]))               # (1,100)
    sims  = cosine_similarity(emb, q_vec).flatten()              # (N,)

    G = nx.DiGraph()
    G.add_node('query')
    for i, code in enumerate(df['courseCode']):
        G.add_edge('query', code, weight=float(sims[i]))

    pr = nx.pagerank(G, alpha=0.85, personalization={'query':1.0})
    ranked = sorted(
        ((c,sc) for c,sc in pr.items() if c!='query'),
        key=lambda x: x[1], reverse=True
    )[:top_k]
    codes  = [c for c,_ in ranked]
    scores = [s for _,s in ranked]
    return df[df['courseCode'].isin(codes)][['courseCode','title','description']].assign(score=scores)



## Method 5: Fuzzy String Matching with Multiple Fields

In [84]:

from difflib import SequenceMatcher

def fuzzy_similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def recommend_fuzzy_multi(query, top_k=10, weights={'title': 0.4, 'description': 0.5, 'code': 0.1}):
    query_lower = query.lower()
    scores = []
    
    for idx, row in df.iterrows():
        title_sim = fuzzy_similarity(query_lower, str(row['title']))
        desc_sim = fuzzy_similarity(query_lower, str(row['description']))
        code_sim = fuzzy_similarity(query_lower, str(row['courseCode']))
        
        # Weighted combination
        total_score = (weights['title'] * title_sim + 
                      weights['description'] * desc_sim + 
                      weights['code'] * code_sim)
        
        scores.append((idx, total_score))
    
    # Sort by score
    scores.sort(key=lambda x: x[1], reverse=True)
    top_indices = [idx for idx, _ in scores[:top_k]]
    top_scores = [score for _, score in scores[:top_k]]
    
    return df.iloc[top_indices][['courseCode','title','description']].assign(fuzzy_score=top_scores)


# Method 6: Keyword Overlap with Position Weighting

In [85]:

def extract_keywords(text, min_length=3):
    """Extract keywords, removing stop words and short terms"""
    import re
    stop_words = {'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'this', 'that', 'these', 'those', 'a', 'an'}
    words = re.findall(r'\b\w+\b', text.lower())
    return [w for w in words if len(w) >= min_length and w not in stop_words]

def recommend_keyword_overlap(query, top_k=10):
    query_keywords = set(extract_keywords(query))
    scores = []
    
    for idx, row in df.iterrows():
        title_keywords = set(extract_keywords(str(row['title'])))
        desc_keywords = set(extract_keywords(str(row['description'])))
        
        # Calculate overlaps
        title_overlap = len(query_keywords.intersection(title_keywords))
        desc_overlap = len(query_keywords.intersection(desc_keywords))
        
        # Position-weighted score (title matches worth more)
        total_keywords = len(title_keywords.union(desc_keywords))
        if total_keywords > 0:
            score = (2 * title_overlap + desc_overlap) / np.sqrt(total_keywords)
        else:
            score = 0
            
        scores.append((idx, score))
    
    scores.sort(key=lambda x: x[1], reverse=True)
    top_indices = [idx for idx, _ in scores[:top_k]]
    top_scores = [score for _, score in scores[:top_k]]
    
    return df.iloc[top_indices][['courseCode','title','description']].assign(keyword_score=top_scores)

## Method 7: BERT embeddings

In [86]:
from sklearn.metrics.pairwise import cosine_similarity

# Load BERT embeddings
bert_embeddings = np.load('data/course_bert_embeddings.npy')

def recommend_bert(query, top_k=10):
    query_embed = model.encode([query])
    sims = cosine_similarity(query_embed, bert_embeddings).flatten()
    top_indices = sims.argsort()[::-1][:top_k]
    return df.iloc[top_indices][['courseCode', 'title', 'description']].assign(bert_score=sims[top_indices])


## Method 8: Hybrid Ensemble Approach

In [87]:


def recommend_hybrid_ensemble(query, top_k=10, method_weights=None):
    """Combine multiple methods with weighted voting"""
    if method_weights is None:
        method_weights = {
            'cosine': 0.2,
            'fuzzy': 0.2,
            'keyword': 0.2,
            'faiss': 0.2,
            'bert': 0.2
        }
    
    # Get results from different methods
    cosine_results = recommend_cosine(query, top_k=len(df))  # Get all for scoring
    fuzzy_results = recommend_fuzzy_multi(query, top_k=len(df))
    keyword_results = recommend_keyword_overlap(query, top_k=len(df))
    faiss_results = recommend_faiss(query, top_k=len(df))
    bert_results = recommend_bert(query, top_k=len(df))
    
    # Normalize scores for each method
    def normalize_scores(scores):
        scores = np.array(scores)
        if scores.max() != scores.min():
            return (scores - scores.min()) / (scores.max() - scores.min())
        return scores
    
    # Create combined scores
    combined_scores = {}
    
    for idx, code in enumerate(df['courseCode']):
        score = 0
        
        # Find scores from each method
        cosine_score = cosine_results[cosine_results['courseCode'] == code]['similarity'].iloc[0] if code in cosine_results['courseCode'].values else 0
        fuzzy_score = fuzzy_results[fuzzy_results['courseCode'] == code]['fuzzy_score'].iloc[0] if code in fuzzy_results['courseCode'].values else 0
        keyword_score = keyword_results[keyword_results['courseCode'] == code]['keyword_score'].iloc[0] if code in keyword_results['courseCode'].values else 0
        faiss_score = faiss_results[faiss_results['courseCode'] == code]['similarity'].iloc[0] if code in faiss_results['courseCode'].values else 0
        bert_score = bert_results[bert_results['courseCode'] == code]['bert_score'].iloc[0] if code in bert_results['courseCode'].values else 0
        
        # Weighted combination
        combined_score = (method_weights['cosine'] * cosine_score +
                         method_weights['fuzzy'] * fuzzy_score +
                         method_weights['keyword'] * keyword_score +
                         method_weights['faiss'] * faiss_score +
                         method_weights['bert'] * bert_score)
        
                        
        
        combined_scores[idx] = combined_score
    
    # Sort and return top results
    sorted_indices = sorted(combined_scores.keys(), key=lambda x: combined_scores[x], reverse=True)
    top_indices = sorted_indices[:top_k]
    top_scores = [combined_scores[idx] for idx in top_indices]
    
    return df.iloc[top_indices][['courseCode','title','description']].assign(hybrid_score=top_scores)

## Testing

In [89]:
search_queries = [
    "machine learning algorithms and data science",
    "analyze financial statements using Python and Excel",
    "a course about dinosaurs, fossils and ancient civilizations"
]

methods = [
    ("cosine", recommend_cosine),
    ("faiss", recommend_faiss),
    ("mmr", recommend_mmr),
    ("graph", recommend_graph),
    ("fuzzy_multi", recommend_fuzzy_multi),
    ("keyword_overlap", recommend_keyword_overlap),
    ("bert", recommend_bert),
    ("hybrid_ensemble", recommend_hybrid_ensemble)
]

# Prepare empty lists for dataframes to be filled
df_list = [[], [], []]

for i, search_query in enumerate(search_queries):
    for method_name, method_func in methods:
        try:
            results = method_func(search_query, top_k=10)
            # results is a DataFrame with courseCode, title, description columns
            for rank, (_, row) in enumerate(results.iterrows(), 1):
                # Check if score column exists in index (columns)
                score_col = None
                for col in ['similarity', 'fuzzy_score', 'keyword_score', 'bert_score', 'hybrid_score', 'score']:
                    if col in row.index:
                        score_col = col
                        break
                
                df_list[i].append({
                    "search_query": search_query,
                    "method": method_name,
                    "rank": rank,
                    "course_code": row['courseCode'],
                    "title": row['title'],
                    "description": row['description'],
                    "score": row[score_col] if score_col else 0
                })
        except Exception as e:
            print(f"Error with method {method_name} on query '{search_query}': {e}")
            continue

# Convert lists to DataFrames **after** data collection
df1 = pd.DataFrame(df_list[0])
df2 = pd.DataFrame(df_list[1])
df3 = pd.DataFrame(df_list[2])

# Export to Excel with 3 sheets
with pd.ExcelWriter("recommendations_results.xlsx") as writer:
    df1.to_excel(writer, sheet_name="Query1_Results", index=False)
    df2.to_excel(writer, sheet_name="Query2_Results", index=False)
    df3.to_excel(writer, sheet_name="Query3_Results", index=False)

print("Excel file 'recommendations_results.xlsx' created with 3 sheets, one per query.")

Excel file 'recommendations_results.xlsx' created with 3 sheets, one per query.
